In [2]:
"""
RetailPulse — Sales & Revenue Analytics
Author: Jairaghavendra Sridhar

"""
 
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

In [3]:
# 1. Loading the Dataset

In [6]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
print(f" Loaded {len(df)} rows, {len(df.columns)} columns")

 Loaded 9994 rows, 21 columns


In [14]:
df.head(10)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
5,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694
6,7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800,4,0.00,1.9656
7,8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152
8,9,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040,3,0.20,5.7825
9,10,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000,5,0.00,34.4700


In [7]:
# 2. Cleaning the Column Names

In [9]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_"))

In [10]:
# 3. Fixing the Data Types

In [15]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"]  = pd.to_datetime(df["ship_date"])
df["postal_code"] = df["postal_code"].astype(str).str.zfill(5)

In [16]:
# 4. Feature Engineering

In [18]:
# Date features
df["order_year"]    = df["order_date"].dt.year
df["order_month"]   = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter
df["order_month_name"] = df["order_date"].dt.strftime("%b")
 
# Shipping duration (days)
df["ship_duration_days"] = (df["ship_date"] - df["order_date"]).dt.days
 
# Financial metrics
df["revenue"]        = df["sales"].round(2)
df["cost"]           = (df["sales"] - df["profit"]).round(2)
df["profit_margin"]  = ((df["profit"] / df["sales"]) * 100).round(2)
df["discounted_flag"] = (df["discount"] > 0).astype(int)
 
# Revenue bands for segmentation
df["revenue_band"] = pd.cut(
    df["sales"],
    bins=[0, 100, 500, 1000, 5000, 25000],
    labels=["0-100", "100-500", "500-1K", "1K-5K", "5K+"])

In [19]:
# 5. Validate
print(f"\n Data Summary:")
print(f"   Date range  : {df['order_date'].min().date()} → {df['order_date'].max().date()}")
print(f"   Total revenue : ${df['revenue'].sum():,.2f}")
print(f"   Total profit  : ${df['profit'].sum():,.2f}")
print(f"   Avg margin    : {df['profit_margin'].mean():.2f}%")
print(f"   Nulls after cleaning: {df.isnull().sum().sum()}")


 Data Summary:
   Date range  : 2014-01-03 → 2017-12-30
   Total revenue : $2,297,200.65
   Total profit  : $286,397.02
   Avg margin    : 12.03%
   Nulls after cleaning: 0


In [20]:
# 6. Exporting the Cleaned Dataset

In [21]:
output_path = "superstore_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"\n Cleaned data saved to: {output_path}")
print(f"   Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n Final columns:")
for col in df.columns:
    print(f"   - {col}")
 


 Cleaned data saved to: superstore_cleaned.csv
   Final shape: 9994 rows × 31 columns

 Final columns:
   - row_id
   - order_id
   - order_date
   - ship_date
   - ship_mode
   - customer_id
   - customer_name
   - segment
   - country
   - city
   - state
   - postal_code
   - region
   - product_id
   - category
   - sub_category
   - product_name
   - sales
   - quantity
   - discount
   - profit
   - order_year
   - order_month
   - order_quarter
   - order_month_name
   - ship_duration_days
   - revenue
   - cost
   - profit_margin
   - discounted_flag
   - revenue_band
